In [ ]:
GIT_REPOSITORY = "https://github.com/draklowell/learnable-wavelets.git"
GIT_COMMIT = "main"

In [ ]:
# Fix for invalid sympy dependency on Google Colab
%pip uninstall -y sympy
%pip install --no-cache-dir -q sympy==1.13.3

In [ ]:
%pip install --no-cache-dir -q "learnable-wavelets @ git+{GIT_REPOSITORY}@{GIT_COMMIT}"

In [ ]:
import torch
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from learnable_wavelets import (
    WaveletTransformParameters,
    WaveletTransformParameters2D,
    WaveletTransformAnalysis2D,
    WaveletTransformSynthesis2D,
    plots as lw_plots,
)

In [ ]:
class WaveletTransform(torch.nn.Module):
    def __init__(
        self,
        support_size: int,
        padding_mode: str = "reflect",
    ):
        super().__init__()
        self.params = WaveletTransformParameters(support_size=support_size)
        self.params2d = WaveletTransformParameters2D()
        self.analysis = WaveletTransformAnalysis2D(padding_mode=padding_mode)
        self.synthesis = WaveletTransformSynthesis2D()

    def forward(self, x):
        filters = self.params()
        filters2d = self.params2d(filters).to(dtype=x.dtype)

        decomposition = list(self.analysis(x, filters2d))

        x_rec = self.synthesis(*decomposition, filters2d)

        for i, component in enumerate(decomposition):
            decomposition[i] = component.view(x.shape[0], x.shape[1], -1)

        return (
            torch.concat(decomposition, dim=-1),
            x_rec[:, :, : x.shape[-2], : x.shape[-1]],
        )

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32
param_dtype = torch.float64

transform = transforms.ToTensor()
train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(
    train_ds, batch_size=60000, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available()
)

model = WaveletTransform(
    support_size=6,
    padding_mode="reflect",
).to(device=device, dtype=param_dtype)
compiled_model = torch.compile(model, mode="max-autotune")

optimizer = torch.optim.Adam(compiled_model.parameters(), lr=1e-3)

In [ ]:
def train_step(model, batch, optimizer, lmbda=1e-4):
    model.train()
    x, _ = batch

    x = x.to(device=device, dtype=dtype)

    optimizer.zero_grad()

    coeffs, x_rec = model(x)

    mse_loss = torch.mean(torch.abs(x_rec - x) ** 2)
    l1_loss = torch.mean(torch.abs(coeffs))

    loss = mse_loss + lmbda * l1_loss

    loss.backward()
    optimizer.step()

    return {
        "loss": loss.item(),
        "reconstruction_loss": mse_loss,
        "sparsity_loss": l1_loss,
    }


for epoch in range(250):
    running = {"loss": 0.0, "reconstruction_loss": 0.0, "sparsity_loss": 0.0}

    for batch in train_loader:
        stats = train_step(compiled_model, batch, optimizer)

        for k in running:
            running[k] += stats[k]

    n = len(train_loader)
    print(
        f"Epoch {epoch+1} | "
        f"loss={running['loss']/n:.6f} | "
        f"recon={running['reconstruction_loss']/n:.6f} | "
        f"sparse={running['sparsity_loss']/n:.6f}"
    )

In [ ]:
x, _ = next(iter(train_loader))
x = x[0:1].to(device, dtype=dtype)
coeffs, x_rec = compiled_model(x)

mse = torch.mean(torch.abs(x_rec - x) ** 2)
print(f"MSE: {mse.item():.6f}")

fig, ax = plt.subplots(2, 2, figsize=(15, 15))

ax[0, 0].set_title("Original")
ax[0, 0].imshow(
    x[0, 0].real.cpu().reshape(28, 28).to(torch.float64).detach(), cmap="gray"
)

ax[0, 1].set_title("Reconstructed")
ax[0, 1].imshow(
    x_rec[0, 0].real.cpu().reshape(28, 28).to(torch.float64).detach(), cmap="gray"
)

lw_plots.plot_wavelet(
    model.params().cpu().detach(),
    axes=(ax[1, 0], ax[1, 1]),
    J=3,
)

plt.show()

In [ ]:
import json

result = {}
for name, value in compiled_model.named_parameters():
    name = name.removeprefix("_orig_mod.")
    result[name] = value.cpu().to(torch.float64).detach().tolist()

print(json.dumps(result, indent=4))